# Relational Databases & Caching

Azure offers three deployment shapes for SQL Server, two flexible-server PaaS engines for PostgreSQL and MySQL, and a managed Redis as the canonical cache. They sit on a continuum from "fully managed PaaS with sharp limits" to "raw VM you operate yourself" — and the decision among them is the most consequential one in any database-backed workload.

The shape to hold in your head: **Azure SQL Database** is the cloud-native PaaS — narrow surface, deepest automation, best for greenfield. **Azure SQL Managed Instance** is the lift-and-shift bridge — almost full SQL Server compatibility on managed infrastructure. **SQL on a VM** is full control, full ops. The open-source flexible servers cover PostgreSQL and MySQL in the same PaaS spirit. And **Azure Cache for Redis** is the data-layer accelerator that takes load off all of the above.

## Azure SQL Database — the cloud-native PaaS

**Azure SQL Database** runs a *single logical database* on a multi-tenant SQL Server fleet that Microsoft fully manages. You write T-SQL; Microsoft handles backups, patching, HA, and zone redundancy.

Two purchase models you pick between (and can switch later):

- **vCore-based** — explicit CPU, memory, IO, storage knobs. Maps to hardware shapes (Standard-series, Memory-optimised, Hyperscale). Cleaner mental model; supports Azure Hybrid Benefit and Reserved Capacity discounts.
- **DTU-based** — opaque "database transaction units" bundling CPU/IO/memory into one number. Simpler, older. Microsoft is steering everyone toward vCore.

Three service tiers in vCore:

- **General Purpose** — separated compute and storage; standard latency. The default.
- **Business Critical** — local SSD, Always On AG replicas inside the SLA, lowest latency. For OLTP that cannot wait on remote storage.
- **Hyperscale** — log-streaming architecture that scales storage to 100 TB and supports many readable replicas with near-instant restore from snapshots. The right choice when you outgrow normal SQL or need fast point-in-time restore on huge databases.

**Serverless** is a billing flag on the General Purpose tier: auto-pause when idle, autoscale compute between min and max vCores. Pay per vCore-second when active, pay nothing for storage-only during pause. Use it for dev/test, intermittent apps, and per-tenant databases that don't always run.

**Elastic pools** group many databases into a shared resource budget. When one database is busy, it uses the headroom others aren't using — pay one pool bill, get N databases. The right fit for SaaS apps where each customer gets a database but no individual customer is hot all the time.

## HA, geo-replication, and failover groups

Inside a region, every Azure SQL Database tier gives you an HA SLA. General Purpose uses zone-redundant storage; Business Critical and Hyperscale ship readable replicas across availability zones inside the tier.

Across regions, two features stack:

- **Active geo-replication** — up to four secondary databases in other regions, kept current via asynchronous log shipping. Secondaries are readable. You initiate the failover.
- **Auto-failover groups** — a managed wrapper on top of geo-replication that adds a **read-write listener** and a **read-only listener** (single hostnames whose DNS follows the failover) and an **automatic failover policy**. The application connects to the listener rather than a region-specific endpoint.

Failover is *not free of data loss*. The replication is async; RPO is typically seconds but can be longer under load. For workloads needing zero data loss on regional failure, you have to use a synchronous pattern at the application layer or accept the trade.

AWS comparison: Azure SQL Database General Purpose ≈ RDS for SQL Server (Multi-AZ); Hyperscale ≈ Aurora's architecture (compute + log + page servers); auto-failover groups ≈ RDS multi-region failover with the listener built in.

## Azure SQL Managed Instance

**Azure SQL Managed Instance (MI)** is the answer to "we have a SQL Server today and want a PaaS, but our apps use cross-database queries / SQL Agent jobs / CLR / Service Broker / Linked Servers and we cannot rewrite them."

MI gives you the **full SQL Server instance surface** — multiple databases under one logical instance, SQL Agent, cross-database queries, Service Broker, Database Mail — running on managed infrastructure inside a VNet subnet you choose. Backups, patching, and HA are Microsoft's job; instance-level features behave like classic SQL Server.

Two tiers:

- **General Purpose** — remote storage, lower IO ceiling, the cheaper default.
- **Business Critical** — local SSD + AG replicas, OLTP-grade latency, in-memory OLTP support.

The trade-offs vs Azure SQL Database: longer provision/scale times, no Hyperscale tier (yet), and a per-instance pricing model that is cheaper at scale (many databases per instance) but pricier for one small database. The trade-off vs SQL on VM: you give up sysadmin access; you get back patching and HA.

When to pick MI: **migrating an existing SQL Server estate** where rewriting away from instance-level features is more expensive than paying for MI.

## SQL Server on a VM

**SQL Server on Azure VMs** is the IaaS option. You install SQL Server on a Windows or Linux VM (or, more often, deploy from a Marketplace image that pre-installs it), attach Premium SSD v2 disks, and you have a SQL Server you fully control.

Reach for it when:

- You need a feature MI doesn't yet support (Distributed Transactions across heterogeneous boxes, certain CLR scenarios, very specific version requirements).
- You need full sysadmin access for an ISV product that demands it.
- Licensing economics work better with Azure Hybrid Benefit on Windows + SQL.

Avoid it when MI or Azure SQL Database fits — every day of sysadmin work you sign up for is a day you don't get back.

AWS comparison: SQL on VM ≈ self-managed SQL Server on EC2; MI ≈ RDS Custom for SQL Server (closest analogue, with caveats); Azure SQL Database ≈ regular RDS SQL Server or Aurora-style at the Hyperscale tier.

## Azure Database for PostgreSQL and MySQL — Flexible Server

For open-source databases, Azure's current direction is **Flexible Server** for both **PostgreSQL** and **MySQL**. The older Single Server offerings are deprecated.

Flexible Server gives you:

- **Burstable / General Purpose / Memory Optimised** compute tiers (the standard Azure tier shape).
- **Zone-redundant HA** — synchronous standby in a different AZ with automatic failover. Optional, costs a second instance.
- **Read replicas** in the same or other regions, asynchronous, up to five. Use for read scaling and DR.
- **VNet integration** — the server gets a private IP in a delegated subnet; no public endpoint required.
- **Maintenance window** controls — you pick when patches roll, not Microsoft.
- **PostgreSQL specifics** — extensions like `pg_stat_statements`, `pgvector`, `postgis`; logical replication; major-version in-place upgrades.
- **MySQL specifics** — 5.7 and 8.0 versions; data-in replication for migration; GTID-based replication.

**Cosmos DB for PostgreSQL** (formerly Hyperscale Citus) is the distributed-PostgreSQL offering for sharded workloads; covered in the NoSQL notebook because of its multi-node architecture.

AWS comparison: Flexible Server ≈ RDS for PostgreSQL / MySQL with Multi-AZ; Cosmos DB for PostgreSQL ≈ Aurora PostgreSQL when sharded with Citus.

In [ ]:
# Provision Azure SQL Database with an auto-failover group.

RG=rg-data-demo
PRIMARY=eastus
SECONDARY=westus
az group create -n $RG -l $PRIMARY

# 1. Logical SQL servers in two regions.
az sql server create -g $RG -n sql-app-prod-eus -l $PRIMARY \
  --admin-user dba --admin-password "<strong-pwd>"
az sql server create -g $RG -n sql-app-prod-wus -l $SECONDARY \
  --admin-user dba --admin-password "<strong-pwd>"

# 2. Primary database — vCore General Purpose, zone-redundant.
az sql db create -g $RG -s sql-app-prod-eus -n appdb \
  --service-objective GP_S_Gen5_2 --backup-storage-redundancy Zone \
  --zone-redundant true

# 3. Auto-failover group binding the two servers; primary listener.
az sql failover-group create -g $RG -s sql-app-prod-eus \
  --partner-server sql-app-prod-wus -n fg-app-prod \
  --failover-policy Automatic --grace-period 1 \
  --add-db appdb

# 4. App connects to a single listener hostname; failover is transparent.
echo "Connect to fg-app-prod.database.windows.net (read-write)"
echo "Connect to fg-app-prod.secondary.database.windows.net (read-only)"

## Azure Cache for Redis

**Azure Cache for Redis** is managed Redis. Microsoft runs the nodes, patches them, fails them over; you point your application at a hostname and a key.

Four tiers, with very different operational profiles:

- **Basic** — single node, no SLA. Dev/test only.
- **Standard** — primary + replica, two-node SLA. Production-ready for small caches.
- **Premium** — clustering (shard across nodes), VNet injection, Redis persistence (RDB + AOF), zone redundancy. The classic production tier.
- **Enterprise / Enterprise Flash** — built on Redis Inc.'s software, supports Redis modules (Search, JSON, Bloom), active-active geo-replication, 99.999% SLA. For the highest-scale or multi-region workloads.

Two features that bend the design:

- **Clustering** — Premium and Enterprise shard the keyspace across nodes; clients use the cluster-aware Redis driver. Throughput scales with node count. You give up multi-key transactions across slots.
- **Geo-replication** — Premium offers asynchronous, manual-failover geo-replication; Enterprise offers active-active (multi-primary). Active-active is what gets you concurrent writes from multiple regions.

AWS comparison: Standard/Premium ≈ ElastiCache for Redis cluster mode disabled/enabled; Enterprise Active-Active ≈ MemoryDB or ElastiCache Global Datastore (close, not identical).

## Cache patterns

Redis is fast and easy — and the easiest way to introduce bugs into a working system. Pick a pattern deliberately:

- **Cache-aside (lazy loading)** — the application reads the cache; on miss, it reads the DB, writes the value to the cache with a TTL, and returns. Simple, popular, and the source of the **thundering-herd** bug: when a hot key expires, many concurrent requests miss simultaneously and all hit the DB. Mitigate with **probabilistic early refresh** (refresh a key with small probability when nearing TTL) or per-key locking.
- **Write-through** — every write goes to cache *and* DB synchronously. Cache is always consistent; writes pay double latency. Good for write-light, read-heavy data.
- **Write-behind (write-back)** — writes go to cache first; an async worker drains them to the DB. Lowest write latency, but you can lose writes if the cache crashes before drain. Risky for systems of record.
- **Read-through** — variant of cache-aside where the cache itself fetches from the DB on miss (Redis modules or an application abstraction). Hides the miss path from callers.

Two non-cache uses worth knowing because they appear in every Azure architecture diagram:

- **Session store** — App Service and Web App scale-out lose sticky session affinity easily; storing sessions in Redis lets any instance serve any request.
- **Rate limiter / token bucket** — atomic `INCR` + TTL is the smallest possible distributed rate-limit primitive.

Always set a TTL on cache entries unless you have an explicit reason not to. Cache without TTL fills, evicts unpredictably, and becomes a debugging nightmare.

## Choosing the right database

A short decision tree for relational + caching:

```
New app, SQL Server stack?
 └── Azure SQL Database (vCore, Business Critical or Hyperscale)

Existing SQL Server estate to migrate?
 ├── App uses instance-level features → Azure SQL Managed Instance
 └── App fits a single DB / no MI-specific features → Azure SQL Database

Need full sysadmin or unsupported feature?
 └── SQL Server on Azure VM

Open-source app?
 ├── PostgreSQL → Azure Database for PostgreSQL Flexible Server
 └── MySQL    → Azure Database for MySQL Flexible Server

Need a cache?
 ├── Single region, classic patterns → Azure Cache for Redis Premium
 └── Multi-region writes, modules    → Azure Cache for Redis Enterprise
```

And the meta-pattern: a typical workload pairs **Azure SQL Database (or PostgreSQL Flexible Server)** with **Azure Cache for Redis Premium**, fronted by an application that reads-through the cache for hot data and falls back to the database for cold. With auto-failover groups on the DB and zone-redundant Redis, the data tier survives both AZ and regional failures with minutes of effort, not weeks.